In [ ]:
# ── 第一格：环境准备（D02 封装之后，每本 notebook 开头都是这十几行）──
%load_ext autoreload
%autoreload 2
# 改了 src/ 里的 .py，notebook 不用重启就能用上新代码。
from signal_mysql import database as mysql
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from signal_insight import config, geo, quality, viz
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT    # 配色常量
viz.apply_theme()                           # 字体、配色、坐标轴样式：一行
con = mysql.connect(config.DB_SCHEMA)       # 路径从包里来：notebook 在哪个文件夹打开都找得到库
DAY = config.ANALYSIS_DAY                   # 分析日期（口径常量）
NB = "D06_驻留识别"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"    # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)

In [ ]:
from signal_mysql import database as mysql
# 【跟丢了才运行】D1–D5 没做完的同学：把 False 改成 True 再运行，从检查点库复制标准表
CATCH_UP = False
if CATCH_UP:
    with mysql.connect(config.CHECKPOINT_SCHEMA) as ck:
        for t in ["dwd_signal", "dim_cell", "dim_hotspot"]:
            mysql.write_frame(mysql.read_frame(f"SELECT * FROM {t}", ck), t, con, if_exists="replace", index=False)
            print("已恢复", t)

In [ ]:
from signal_mysql import database as mysql
hot = mysql.read_frame("SELECT * FROM dim_hotspot", con).iloc[0]       # 西塔中心（D4）
DAY_START = pd.Timestamp(DAY)                                      # 2018-10-03 00:00

def mask_id(imsi):
    """展示用户时只显示 IMSI 尾号，不在图表和报告里暴露完整 ID。"""
    return "尾号" + str(imsi)[-4:]

In [ ]:
from signal_mysql import database as mysql
dwd = mysql.read_frame("SELECT imsi, ts, lac_cell FROM dwd_signal", con, parse_dates=["ts"])
dim = mysql.read_frame("SELECT lac_cell, site_id, lng, lat FROM dim_cell", con)
sig = (dwd.merge(dim, on="lac_cell", how="inner")                   # 关联不上坐标的 8 条不参与（D3 的决定）
          .sort_values(["imsi", "ts"], ignore_index=True))
sig["到西塔(米)"] = geo.haversine(sig["lat"], sig["lng"], hot["lat"], hot["lng"])
n_users = sig["imsi"].nunique()
print(f"带坐标的信令 {len(sig):,} 条，用户 {n_users:,} 个")

In [ ]:
h = sig["ts"].dt.hour
night_far = sig[(h == 3) & (sig["到西塔(米)"] > 3000)]["imsi"].unique()           # 凌晨 3 点在 3 公里外
evening_in = sig[(h.between(19, 20)) & (sig["到西塔(米)"] <= 500)]["imsi"].unique()   # 19–20 点在西塔 500 米内
n_rec = sig.groupby("imsi").size()
candidates = n_rec[n_rec.index.isin(night_far) & n_rec.index.isin(evening_in) & n_rec.between(60, 120)]
demo_id = candidates.index[0]                                          # 按 IMSI 排序取第一个，保证每次选到同一个人
demo = sig[sig["imsi"] == demo_id]
print(f"候选 {len(candidates)} 人；示例用户 {mask_id(demo_id)}，当天 {len(demo)} 条信令")

In [ ]:
hour = (demo["ts"] - DAY_START).dt.total_seconds() / 3600           # 时间换算成"几点"（小数）
km = demo["到西塔(米)"] / 1000
arrive = demo[(demo["ts"].dt.hour >= 12) & (demo["到西塔(米)"] <= 500)]["ts"].min()
night_km = km[demo["ts"].dt.hour < 6].median()
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.axhspan(0, 0.5, color=MAIN_LIGHT, lw=0, zorder=0)                      # 西塔 500 米范围
ax.text(0.2, 0.25, "西塔 500 米内", va="center", color=MAIN, fontsize=9.5)
ax.plot(hour, km, color=GRAY, lw=1, zorder=1)
ax.scatter(hour, km, s=16, color=MAIN, zorder=2)
ax.annotate(f"{arrive:%H:%M} 到达西塔", (hour[demo["ts"] == arrive].iloc[0], 0.5), xytext=(25, 70),
            textcoords="offset points", ha="left", fontsize=10, color=INK, arrowprops=dict(arrowstyle="->", color=FAINT))
ax.set_xticks(range(0, 25, 3), [f"{x} 点" for x in range(0, 25, 3)])
ax.set_xlim(0, 24)
ax.set_ylim(0, km.max() * 1.15)
ax.set_ylabel("离西塔的距离（公里）", color=MUTED)
title = f"{mask_id(demo_id)} 的一天：凌晨在距西塔约 {night_km:.1f} 公里处，{arrive:%H:%M} 到达西塔"
viz.finish(fig, title,
           f"2018-10-03 该用户每条信令所连站点到西塔中心的距离；共 {len(demo)} 条；口径：dwd_signal × dim_cell",
           SOURCE, "D6_01_一个人的一天.png")

In [ ]:
# ════ 本日第一个关键概念：乒乓切换 ════
# 这一格在做什么：量一量"换了站点"这件事到底有多频繁、换的距离有多远。
#
# 手机连哪个基站，取决于**信号强弱**，不取决于人在不在动。人坐着不动，
# 信号稍有起伏就会在相邻两座基站之间来回切（A → B → A），这就是乒乓切换。
# 下面三个数把它坐实了：换站点很频繁、换的距离很近、而且有相当比例是 10 分钟内又切回来。
#
# 讲台提示：不理解这件事，就理解不了 6.3 为什么要"按距离切段"而不是"按基站分组"。
# 先问学生："基站变了，是不是说明人动了？"大部分人会说是。
g = sig.groupby("imsi")
prev_site = g["site_id"].shift()                                                   # 本人上一条记录的站点
switch = prev_site.notna() & (sig["site_id"] != prev_site)                         # 和上一条相比换了站点
jump_m = geo.haversine(sig["lat"], sig["lng"], g["lat"].shift(), g["lng"].shift())

In [ ]:
switch_m = pd.Series(jump_m[switch.to_numpy()])
# A → B → A 的判定：当前这条换了站点（switch），而且**下一条又切回了上上条的站点**，
# 并且上一条到下一条之间不超过 10 分钟。shift(-1) 取下一条、shift() 取上一条，
# 两个方向都要先 groupby("imsi")，否则会跨人取到别人的记录。
aba = switch & (g["site_id"].shift(-1) == prev_site) & \
      ((g["ts"].shift(-1) - g["ts"].shift()).dt.total_seconds() <= 600)          # 10 分钟内 A→B→A
print(f"相邻记录换了站点：{switch.mean():.1%}")
print(f"换站点时两站距离：中位 {switch_m.median():.0f} 米，{(switch_m <= 1000).mean():.0%} 在 1 公里以内")
print(f"10 分钟内又切回原站点（A → B → A）：占全部切换的 {aba.sum() / switch.sum():.1%}")

In [ ]:
# ── 出图 D6_02：直方图 ────────────────────────────────────────────────
# 为什么用直方图而不是 ECDF：这里要让读者看见**形状**——绝大多数切换挤在最左边一两公里内，
# 右边拖一条长尾。形状本身就是"人没动、基站在变"的证据。
# （D4_03 要回答"多少人在 500 米内"这种累计问题，那里才用 ECDF。两者的分工就在这。）
# 规范落在哪几行：
#   · 纵轴用**占比**不用计数 —— 换一天数据、样本量变了，图的高度还能横着比
#   · clip(upper=4999) 把 5 公里以上并进最后一组，横轴标签写明"最后一组含 5 公里以上" ——
#     长尾不截断的话，右边 90% 的画布全是空白
#   · 1 公里以内上主色、以外灰色 —— 把"要强调的那部分"直接涂出来，不靠读者自己判断
# 标题里的数字：switch.mean()（换站点的比例）、p50（切换距离的中位数）。
bins = np.arange(0, 5001, 250)
counts, _ = np.histogram(switch_m.clip(upper=4999), bins=bins)
share = counts / counts.sum()
p50 = switch_m.median()
fig, ax = plt.subplots(figsize=(10, 5.6))
colors = [MAIN if b < 1000 else GRAY for b in bins[:-1]]
ax.bar(bins[:-1] / 1000, share, width=0.23, align="edge", color=colors)
ax.axvline(p50 / 1000, color=INK, lw=1, ls=(0, (4, 3)))
ax.text(p50 / 1000, share.max() * 1.02, f" 中位数 {p50:,.0f} 米", color=INK, fontsize=10)
ax.set_xticks(range(0, 6), [f"{x} 公里" if x else "0" for x in range(0, 6)])
ax.set_xlabel("切换前后两个站点的距离（最后一组含 5 公里以上）", color=MUTED)
ax.yaxis.set_major_formatter("{x:.0%}")
ax.set_ylim(0, share.max() * 1.15)
title = f"{switch.mean():.0%} 的相邻信令换了站点，一半的切换距离不到 {p50:,.0f} 米"
viz.finish(fig, title,
           "站点切换距离的分布，每 250 米一组；蓝色 = 1 公里以内，多为人没动时相邻基站之间的切换；口径：dwd_signal × dim_cell",
           SOURCE, "D6_02_站点切换距离.png")

In [ ]:
# 对账
assert 0.44 < switch.mean() < 0.46 and 600 < p50 < 700
print(f"✅ 对账通过：{switch.mean():.1%} 的相邻记录换了站点，切换距离中位 {p50:.0f} 米")

In [ ]:
import math
import time

def dist_m(lat1, lng1, lat2, lng2):
    """两点距离（米），标量版：循环里一次只算一对点，用 math 比用 numpy 快。"""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lng2 - lng1) / 2) ** 2
    return 2 * 6_371_000 * math.asin(math.sqrt(a))

# ════ 全程最核心的算法，务必讲透 ════
#
# 思路：按时间顺序往下走，维护"当前这一段"的**质心**（已并入各点的平均位置）。
#   下一条信令离质心 ≤ 500 米 → 并进来，质心跟着更新；
#   > 500 米 → 当前段结束，从这一条开始新的一段。
# 段结束时记下：谁、起止时刻、质心坐标、并了几条。段长 ≥ 30 分钟的，才算一次"停留"。
#
# 为什么比到**质心**，不是比到**上一个点**：
#   比到上一个点的话，一串每次都走 400 米的信令会被一直并成同一段，
#   走出十几公里也不会断开（"温水煮青蛙"）。比到质心就不会——离起点越远，差距越明显。
#
# 讲台提示：画在白板上最有效。横轴时间、纵轴位置，点一个个往右画，
# 每并进一个点就把质心往那边挪一点，直到有个点跳出 500 米圈。
#
# 两个必须守住的前提（D06 的契约测试里有对应的两条）：
#   ① df 必须先按 imsi、ts 排好序（D2 写 dwd_signal 时已经排过）；
#   ② 内层 while 的 imsi[j + 1] == imsi[i] 保证**不跨用户分段**——
#      少了这个判断，上一个人的最后一段会接上下一个人的第一条。
#
# 为什么全程用 numpy 数组而不是 DataFrame：循环里每次取值，
# df.iloc[i] 比 arr[i] 慢上百倍。50 万行的循环，这个差别是"几秒"和"十几分钟"。
def find_segments(df, radius_m=500):
    """把每个人按时间排好的信令切成"段"：离当前段质心不超过 radius_m 就并入，否则开新段。"""
    imsi = df["imsi"].to_numpy()
    t = (df["ts"] - pd.Timestamp("1970-01-01")).dt.total_seconds().to_numpy()   # 秒，和 pandas 版本无关
    lat, lng = df["lat"].to_numpy(), df["lng"].to_numpy()
    rows, i, n = [], 0, len(df)
    while i < n:
        sum_lat, sum_lng, k, j = lat[i], lng[i], 1, i
        while j + 1 < n and imsi[j + 1] == imsi[i]:
            if dist_m(sum_lat / k, sum_lng / k, lat[j + 1], lng[j + 1]) > radius_m:
                break
            j += 1
            sum_lat, sum_lng, k = sum_lat + lat[j], sum_lng + lng[j], k + 1
        rows.append((imsi[i], t[i], t[j], sum_lat / k, sum_lng / k, k))
        i = j + 1
    seg = pd.DataFrame(rows, columns=["imsi", "t_start", "t_end", "lat", "lng", "n_records"])
    seg["duration_min"] = (seg["t_end"] - seg["t_start"]) / 60
    return seg

t0 = time.perf_counter()
seg = find_segments(sig, radius_m=500)
stay = seg[seg["duration_min"] >= 30].reset_index(drop=True)
print(f"{len(sig):,} 条信令 → {len(seg):,} 个段 → {len(stay):,} 个停留（≥30 分钟），用时 {time.perf_counter() - t0:.1f} 秒")
print(f"有停留的用户：{stay['imsi'].nunique():,} 人（{stay['imsi'].nunique() / n_users:.1%}），人均 {len(stay) / n_users:.2f} 个停留")

In [ ]:
# 对账
assert len(seg) == 144_239 and len(stay) == 19_015
print("✅ 对账通过：144,239 个段，19,015 个停留")

In [ ]:
from signal_mysql import database as mysql
# 这一格在做什么：敏感性分析——把半径和最短时长各试几个值，看结果变多少。
#
# 为什么必须做：500 米、30 分钟这两个数不是算出来的，是**选**出来的。
# 答辩时客户一定会问"为什么是 500 米"。能接住这个问题的回答只有一种：
# "我试过 300 到 1000 米，人均停留点只从 3.4 变到 3.2，结论对这个参数不敏感。"
#
# 讲台提示：企业里改参数要能说出依据，这是本课程教的"可辩护的参数选择"。
# 先让学生猜"半径放宽一倍，停留点会多还是少、变化大不大"，再跑。
grid = []
for r in [300, 500, 800, 1000]:
    s = find_segments(sig, radius_m=r)
    for t_min in [15, 30, 60]:
        grid.append({"radius_m": r, "min_minutes": t_min,
                     "stays_per_user": (s["duration_min"] >= t_min).sum() / n_users})
grid = pd.DataFrame(grid)
mysql.write_frame(grid, "ads_stay_sensitivity", con, if_exists="replace", index=False)
grid.pivot(index="radius_m", columns="min_minutes", values="stays_per_user").round(2)

In [ ]:
from signal_mysql import database as mysql
# ── 出图 D6_03：多条折线，选用的那条加粗并圈出 ───────────────────────
# 为什么用折线不用表格：敏感性分析要让人一眼看出"曲线是平的还是陡的"。
# 表格堆一堆数字，读者得自己做减法才知道变化大不大。
# 规范落在哪几行：
#   · 只有选用的 30 分钟那条上主色、加粗，另外两条灰色 —— 一种颜色强调一件事
#   · ax.scatter + annotate 把"选用：500 米、30 分钟"圈出来 —— 让图直接回答"所以你选了哪个"
#   · 线尾 ax.text 标"≥ 15 分钟"等 —— 直接标注，不用图例
#   · set_ylim(0, ...) 纵轴从 0 —— 不从 0 的话三条线会被拉开，显得差异很大，
#     而这张图的结论恰恰是"差异很小"，截断纵轴会把结论讲反
# 标题里的数字：r30[300] 和 r30[1000]，即 30 分钟那条线在最小和最大半径处的取值。
gs = mysql.read_frame("SELECT * FROM ads_stay_sensitivity", con)
fig, ax = plt.subplots(figsize=(10, 5.6))
for t_min, color in ((15, GRAY), (30, MAIN), (60, GRAY)):
    d = gs[gs["min_minutes"] == t_min]
    ax.plot(d["radius_m"], d["stays_per_user"], color=color, lw=2.4 if t_min == 30 else 1.6, marker="o")
    ax.text(d["radius_m"].iloc[-1] + 25, d["stays_per_user"].iloc[-1], f"≥ {t_min} 分钟", va="center",
            color=color if t_min == 30 else MUTED, fontsize=10, fontweight="bold" if t_min == 30 else "normal")
chosen = gs[(gs["radius_m"] == 500) & (gs["min_minutes"] == 30)]["stays_per_user"].iloc[0]
ax.scatter([500], [chosen], s=90, facecolor="white", edgecolor=MAIN, lw=2, zorder=3)
ax.annotate("选用：500 米、30 分钟", (500, chosen),
            xytext=(0, 14), textcoords="offset points", ha="center", fontsize=10, color=INK)
ax.set_xticks([300, 500, 800, 1000], ["300 米", "500 米", "800 米", "1 公里"])
ax.set_xlim(250, 1130)
ax.set_ylim(0, gs["stays_per_user"].max() * 1.2)
ax.set_ylabel("人均停留点（个）", color=MUTED)
r30 = gs[gs["min_minutes"] == 30].set_index("radius_m")["stays_per_user"]
title = f"半径从 300 米放宽到 1 公里，人均停留点只从 {r30[300]:.1f} 个变到 {r30[1000]:.1f} 个"
viz.finish(fig, title,
           "不同半径、不同最短时长下识别出的人均停留点数；结果对半径不敏感，对最短时长较敏感；口径：dwd_signal × dim_cell",
           SOURCE, "D6_03_参数敏感性.png")

In [ ]:
# 业务参数从 settings.yaml 读：这是 D06 起唯一的参数来源
settings = config.load_settings()
settings["stay"], settings["hotspot"]

In [ ]:
# 跑全部测试（项目根目录在上一级）
!python -m pytest ../tests -q

In [ ]:
from signal_mysql import database as mysql
# 收进包的版本和上面一步步算的结果对账
from signal_insight import dws, pipeline
districts = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_district_boundary", con), "district")
aois = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_aoi_v2", con), "name")
dws_stay = dws.build_stay(sig, settings, districts, aois, hot)
seg_pkg = dws.find_segments(sig.sample(frac=1, random_state=0), settings["stay"]["radius_m"])    # 故意打乱顺序再传进去
print(f"包里的版本：{len(seg_pkg):,} 个段，{len(dws_stay):,} 个停留；notebook 版本：{len(seg):,} 个段，{len(stay):,} 个停留")
# 对账：结果与 notebook 版本一致；打乱顺序的输入也一样（函数自己排序，同一时刻按 lac_cell 决胜）
assert len(seg_pkg) == len(seg) == 144_239 and len(dws_stay) == len(stay) == 19_015
assert (dws_stay["duration_min"].to_numpy() == stay["duration_min"].round(1).to_numpy()).all()
print("✅ 对账通过：dws.build_stay 与 notebook 版本一致，19,015 个停留")

In [ ]:
# 流水线多了一步：dws_stay（整表替换）
log = pipeline.run_day()
log[["step", "rows_in", "rows_out", "elapsed_sec"]].tail(4)

In [ ]:
from signal_mysql import database as mysql
from signal_insight import db
# stay_seq：每个人当天的第 1、2、3……个停留。它和 imsi 一起构成这张表的主键，
# D7 算"出行"时就靠它找"下一个停留"（相邻两个停留之间的移动 = 一次出行）。
assert not dws_stay.duplicated(["imsi", "stay_seq"]).any()                    # 主键：imsi + stay_seq
db.replace_table(con, dws_stay, "dws_stay")
print(mysql.read_frame("SELECT COUNT(*) AS 停留数, COUNT(DISTINCT imsi) AS n_users FROM dws_stay", con))
dws_stay.head(3).assign(imsi=dws_stay["imsi"].head(3).map(mask_id))           # 展示时脱敏

In [ ]:
# ── 出图 D6_04：直方图 ────────────────────────────────────────────────
# 为什么用直方图：要让读者看见停留时长是"短的多、长的少，但 8 小时以上有明显一坨"——
# 那一坨就是过夜，D7 整天都在处理它。形状比任何统计量都说明问题。
# 规范落在哪几行：
#   · ≥ 8 小时的部分单独上主色并写比例 —— 把要强调的那部分涂出来
#   · 虚线标中位数，并在旁边写"一半的停留比它长" —— 中位数不解释，读者容易当成平均数
#   · 副标题写"时长 = 段内首条到末条信令，是实际停留的下限" ——
#     **这是口径限制，必须写**：人到了之后第一条信令才出现，走之前最后一条信令就停了，
#     两头都少算。D8.6 的"删失"是同一类问题的另一种形态
# 标题里的数字：len(hrs)（停留总数）、med（中位时长）、long_（≥ 8 小时的占比）。
hrs = dws_stay["duration_min"] / 60
med, long_ = hrs.median(), (hrs >= 8).mean()
cnt_, edges = np.histogram(hrs, bins=np.arange(0.5, 24.01, 0.5)); share_ = cnt_ / len(hrs)
fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
# 直方图（手册 4.4）：纵轴用占比，虚线标中位数，长尾单独上色并写比例
ax.bar(edges[:-1], share_, width=0.46, align="edge", color=[MAIN if e >= 8 else GRAY for e in edges[:-1]])
ax.axvline(med, color=INK, lw=1.2, ls="--")
ax.text(med + 0.25, share_.max() * 0.95, f"中位数 {med:.1f} 小时：\n一半的停留比它长", fontsize=9.5, color=INK, va="top")
ax.text(14, share_.max() * 0.30, f"≥ 8 小时占 {long_:.0%}\n多为过夜（跨过 0–6 点）", fontsize=9.5, color=MAIN,
        fontweight="bold", ha="center")
ax.set_xlim(0.4, 24.2); ax.set_xticks(range(1, 25, 2))
ax.set_xlabel("停留时长（小时，每 0.5 小时一组）")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax.set_ylabel("占全部停留")
title = f"识别出 {len(hrs):,} 个停留，一半超过 {med:.1f} 小时，{long_:.0%} 超过 8 小时"
viz.finish(fig, title,
           "停留 = 500 米内持续 30 分钟以上；时长 = 段内首条到末条信令，是实际停留的下限；蓝色 = 8 小时以上；口径：dws_stay（10-03）",
           SOURCE, "D6_04_停留时长分布.png")

In [ ]:
# 这一格在做什么：用"按基站合并"这种更直觉的做法算一遍，和 6.3 的结果比一比。
# 这是对 6.3 那个算法的**反证**：按基站合并会漏掉约三成的停留，
# 因为乒乓切换把一次长停留切成了许多短段，每段都不够 30 分钟。
#
# cumsum() 这个技巧值得单独讲一分钟：布尔序列累加，每遇到一个 True 编号就 +1，
# 于是"连续相同的一串"自动获得同一个编号。给连续段编号的标准写法。
new_run = sig["imsi"].ne(sig["imsi"].shift()) | sig["site_id"].ne(sig["site_id"].shift())
run_id = new_run.cumsum()                                            # 每换一次人或站点，编号加 1
runs = sig.groupby(run_id).agg(imsi=("imsi", "first"), start=("ts", "min"), end=("ts", "max"))
runs["duration_min"] = (runs["end"] - runs["start"]).dt.total_seconds() / 60
by_site = (runs["duration_min"] >= 30).sum()

In [ ]:
print(f"按站点合并：{len(runs):,} 段，其中 ≥30 分钟的停留 {by_site:,} 个")
print(f"按距离切段：{len(seg):,} 段，其中 ≥30 分钟的停留 {len(stay):,} 个")
print(f"按站点合并会漏掉 {1 - by_site / len(stay):.0%} 的停留：乒乓切换把一次长停留切成了很多短段")

In [ ]:
from signal_mysql import database as mysql
n = mysql.read_frame("""SELECT s.imsi, s.ts, c.site_id FROM dwd_signal s
                   JOIN dim_cell c ON s.lac_cell = c.lac_cell""", con, parse_dates=["ts"])
n = n[n["ts"].dt.hour.between(1, 4)]                                        # 1:00–4:59
per_user = n.groupby("imsi").size()
ns = n[n["imsi"].isin(per_user[per_user >= 5].index)].groupby("imsi")["site_id"].nunique()    # 这段时间经过几个站点
print(f"凌晨有 5 条以上信令的 {len(ns):,} 人：{(ns >= 2).mean():.1%} 经过 2 个以上站点，90 分位 {ns.quantile(.9):.0f} 个")

In [ ]:
# 对账
assert len(ns) == 2_511 and round((ns >= 2).mean(), 3) == 0.626
print("✅ 对账通过：2,511 人，62.6% 凌晨经过 2 个以上站点")

In [ ]:
# ── 出图 D6_05：分组柱状图 ────────────────────────────────────────────
# 为什么把站点数分成不等宽的组（1 / 2 / 3 / 4–5 / 6–10 / …）：
# 原始取值从 1 到几十，等宽分组会让右边一堆组都只有零点几个百分点。
# 按量级分组，每组都看得见，而且"1 个"和"2 个以上"的对比正好是这张图的结论。
# 规范落在哪几行：
#   · "1 个"灰色、其余主色 —— 要强调的是"居然有这么多人超过 1 个"
#   · bar_label 把百分比标在柱顶、再 set_yticks([]) 去掉数值轴 —— 标了数就不要刻度
# 标题里的数字：(ns >= 2).mean()，经过 2 个以上站点的人的占比。
bins_ = pd.cut(ns, [0, 1, 2, 3, 5, 10, 20, np.inf],
               labels=["1 个", "2 个", "3 个", "4–5 个", "6–10 个", "11–20 个", "20 个以上"])
dist_ = bins_.value_counts(sort=False) / len(ns)
fig, ax = plt.subplots(figsize=(10, 5.4))
bars = ax.bar(dist_.index.astype(str), dist_.to_numpy() * 100, color=[GRAY] + [MAIN] * (len(dist_) - 1), width=0.62)
ax.bar_label(bars, labels=[f"{v:.0%}" for v in dist_], padding=4, color=INK)
ax.set_yticks([]); ax.grid(False)
ax.set_xlabel("凌晨 1–5 点经过的站点数", color=MUTED)
title = f"凌晨 1–5 点人在睡觉，{(ns >= 2).mean():.0%} 的人的信令却经过了 2 个以上站点"
viz.finish(fig, title,
           f"凌晨 1:00–4:59 至少有 5 条信令的 {len(ns):,} 人，按经过的站点数分组；一成的人超过 {ns.quantile(.9):.0f} 个站点；口径：dwd_signal × dim_cell",
           SOURCE, "D6_05_凌晨跳站.png")

In [ ]:
con.close()